# 03 — Preparación y feature engineering

**TFM: Predicción de emisiones de CO₂ de buques (THETIS-MRV)**

Punto de partida: `data/processed/mrv_eda.parquet`, el dataset resultante del notebook 02 (EDA
estadístico) — 106.701 filas × 13 columnas, ya filtrado a informes `Full`, con los atípicos
tratados y los predictores imputados.

Este notebook resuelve y aplica tres decisiones de diseño que el EDA deja abiertas, y entrega los
datasets listos para la modelización (notebook 04 en adelante):

1. Cómo tratar las subcategorías nuevas de `ship_type` (cruceros, offshore) introducidas por EMSA
   solo desde 2023-2025.
2. Qué hacer con `total_fuel_consumption_m_tonnes`, que correlaciona 0,96 con el target — casi
   tautológico.
3. Cómo particionar train/test sin que un mismo buque quede a los dos lados.

Además, verifica la fiabilidad del target antes de modelizar y trata los atípicos de `te_valor`,
que en el EDA solo recibe imputación de sus missings directos y conserva una cola de valores
extremos.

Todas las funciones están en `src/feature_engineering.py`, y reutilizan `src/eda_utils.py` para
mantener el mismo criterio de depuración en toda la tubería.

In [1]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from eda_utils import atipicosAmissing
from feature_engineering import (
    reconstruir_target_fiable,
    reconstruir_distancia_velocidad,
    reconstruir_capacidad,
    agrupar_ship_type,
    tratar_atipicos_te_valor,
    dividir_train_test,
    construir_datasets_modelizacion,
    PREDICTORES_PRINCIPAL,
    PREDICTORES_CONTROL,
    PREDICTORES_TAMANO,
    CAPACIDAD_COLS,
    RANDOM_STATE,
)

pd.set_option('display.max_columns', 50)

df = pd.read_parquet('../data/processed/mrv_eda.parquet')
print(f'Dataset de partida: {df.shape[0]:,} filas x {df.shape[1]} columnas'.replace(',', '.'))
df.head(3)


Dataset de partida: 106.701 filas x 13 columnas


,reporting_year,ship_imo_number,ship_name,company_name,total_co2_emissions_m_tonnes,total_fuel_consumption_m_tonnes,time_spent_at_sea_hours,ship_type,ice_class,technical_efficiency,te_metodo,te_valor,prop_missings
0,2018,5383304,ASTORIA,NaN,20080.2509,6307.75,4170.20,Passenger ship,Sin clase de hielo,Not Applicable,Not Applicable,4.52,0.25
1,2018,6417097,MARCO POLO,NaN,25689.0258,8125.56,4360.57,Passenger ship,IC,Not Applicable,Not Applicable,8.65,0.25
2,2018,6511128,RED STAR 1,NaN,6941.3412,2205.37,2712.00,Ro-pax ship,Sin clase de hielo,EIV (45.57 gCO₂/t·nm),EIV,45.57,0.0


## 1. Verificación de la fiabilidad del target antes de modelizar

El CO₂ y el combustible están ligados por una relación física conocida: el CO₂ se define
reglamentariamente como el combustible multiplicado por el factor de emisión del fuel marino
(~3,11-3,21 t CO₂/t; ver notebook 01, validación física). Antes de modelizar conviene comprobar,
sobre el dataset que sale del EDA, que esa relación se mantiene fila a fila — si no se mantiene,
alguna de las dos variables no es de fiar, y una de ellas es precisamente el target.

Imputar CO₂ y combustible **de forma independiente** (cada variable con un sorteo de su propia
distribución empírica, sin mirar el valor de la otra en esa misma fila) rompería la relación física
en ~3.000 filas. Por eso el notebook 02 (sección 6b) no aplica el criterio estadístico de atípicos
al CO₂ ni al combustible y no imputa nunca la variable objetivo; aquí se verifica que efectivamente
no queda ningún caso.

In [2]:
co2 = df['total_co2_emissions_m_tonnes']
fuel = df['total_fuel_consumption_m_tonnes']
ratio = co2 / fuel

print(ratio.describe())
print()
fuera_de_rango = (fuel > 0) & ((ratio < 2.5) | (ratio > 3.3))
ambos_cero = (fuel == 0) & (co2 == 0)
print(f'Filas con ratio fuera de [2.5, 3.3] (fuel>0): {fuera_de_rango.sum()} ({fuera_de_rango.mean():.2%})')
print(f'Filas con CO2 = combustible = 0 (buque inactivo, valor real, no artefacto): {ambos_cero.sum()}')


count    100883.000000
mean          3.132676
std           0.066916
min           2.508379
25%           3.124569
50%           3.136619
75%           3.157126
max           3.288215
dtype: float64

Filas con ratio fuera de [2.5, 3.3] (fuel>0): 0 (0.00%)
Filas con CO2 = combustible = 0 (buque inactivo, valor real, no artefacto): 5638


Verificado: **0 filas** con el ratio fuera de rango (con imputación independiente serían 3.181). La distribución del ratio está concentrada entre 2,51 y 3,29, con mediana 3,137 —
exactamente el factor de emisión del fuel marino. La ruptura de la relación física sería, en su
totalidad, un artefacto de la imputación independiente: no hay ningún problema en el dato
original.

Las 5.638 filas con CO₂ = combustible = 0 son un caso distinto y legítimo: un buque inactivo todo
el año reporta ambas variables en cero de forma consistente. No es un artefacto, se conservan.

Se mantiene aun así el cruce con `mrv_consolidado.parquet` (el dato crudo, antes de que el EDA
tocara nada) como verificación cruzada independiente: identifica qué filas quedaron sin target
fiable y por qué, y reconstruye el valor que falte a partir del de su pareja usando el factor de
emisión físico, cuando solo falta uno de los dos. Con el tratamiento del notebook 02 ese caso no
se da (las 180 filas restantes tienen los dos valores a la vez fuera de la cota física, y no hay
nada de lo que partir para reconstruirlas), pero la comprobación se deja porque es barata y porque
documenta que la exclusión es mínima y está justificada.

In [3]:
df, resumen_reconstruccion = reconstruir_target_fiable(df, '../data/processed/mrv_consolidado.parquet')
print('Factor de emisión usado (mediana, filas sin tratar):', round(resumen_reconstruccion['factor_emision_mediana'], 4))
print()
for clave, valor in resumen_reconstruccion.items():
    print(f'{clave}: {valor}')
print()
print(f"Dataset tras la reconstrucción/filtro: {df.shape[0]:,} filas".replace(',', '.'))


Factor de emisión usado (mediana, filas sin tratar): 3.1366

factor_emision_mediana: 3.1366185983537855
n_completo: 106521
n_reconstruido: 0
n_solo_falta_co2: 0
n_solo_falta_fuel: 0
n_no_fiable: 180

Dataset tras la reconstrucción/filtro: 106.521 filas


Resultado: **106.521 filas fiables** de las 106.701 que entrega el EDA. Solo se excluyen 180
filas (0,17%), las únicas cuya pareja CO₂/combustible es físicamente incompatible en el dato
crudo — un error de reporte que no se puede arreglar sin inventarse un valor. Ninguna fila necesita
reconstrucción: al no aplicarse el criterio estadístico de atípicos sobre estas dos variables,
todas conservan directamente su valor original real.

Aplicar ese criterio dejaría el dataset en 103.401 filas en vez de 106.521 (−3.120), y las
perdidas no serían filas cualesquiera — son las de los mayores emisores de la flota, que concentraban cerca del 19% de todo el CO₂ del dataset (ver
sección 6b del notebook 02).

In [4]:
print('Filas con target reconstruido, por ship_type:')
df.loc[df['target_reconstruido'], 'ship_type'].value_counts()


Filas con target reconstruido, por ship_type:


Series([], Name: count, dtype: int64)

## 2. Taxonomía de `ship_type`: subcategorías nuevas desde 2023-2025

EMSA introdujo dos subcategorías nuevas dentro de tipos ya existentes: **"Passenger ship (Cruise
Passenger ship)"** y **"Other ship types (Offshore)"**. No es un error de los datos ni de la
ingesta — es un cambio real de criterio de clasificación
del propio portal. Antes de usar `ship_type` como variable del modelo hay que decidir cómo
tratarlo, porque tal cual viene puede confundir a un modelo: casi no hay filas de estas
subcategorías antes de 2023-2024, así que un modelo podría aprender "subcategoría rara = año
reciente" en vez de un patrón real de emisiones.


In [5]:
# Evolución anual de las dos subcategorías nuevas frente a su categoría "padre"
cols_interes = ['Passenger ship', 'Passenger ship (Cruise Passenger ship)',
                 'Other ship types', 'Other ship types (Offshore)']
sub = df[df['ship_type'].isin(cols_interes)]
tabla = pd.crosstab(sub['reporting_year'], sub['ship_type'])[cols_interes]
tabla


ship_type,Passenger ship,Passenger ship (Cruise Passenger ship),Other ship types,Other ship types (Offshore)
reporting_year,,,,
2018,159,0,126,0
2019,179,0,142,0
2020,109,0,135,0
2021,113,0,163,0
2022,191,0,257,0
2023,207,10,366,0
2024,211,18,416,6
2025,33,206,414,514


Se confirma el patrón: "Passenger ship (Cruise Passenger ship)" no existe antes de 2023 y pasa a
206 filas en 2025 (mientras "Passenger ship" a secas cae de 211 en 2024 a solo 33 en 2025 —
la caída es casi un trasvase directo a la subcategoría nueva). "Other ship types (Offshore)" es
aún más reciente: prácticamente inexistente hasta 2024 (6 filas) y 514 filas en 2025.

**Decisión (híbrida):**
- `ship_type_agrupado`: se fusionan ambas subcategorías con su categoría padre, para mantener una
  taxonomía estable en toda la serie 2018-2025.
- `es_subcategoria_nueva`: variable binaria que conserva la información de qué filas pertenecían a
  la subcategoría nueva, para poder estudiar su efecto propio (por ejemplo, si los cruceros tienen
  un perfil de emisión distinto por su carga hotelera) en la interpretabilidad (notebook 07), sin
  distorsionar la variable categórica principal con clases casi vacías en años anteriores.


In [6]:
df = agrupar_ship_type(df)

print(f"Filas marcadas como subcategoría nueva: {df['es_subcategoria_nueva'].sum()}")
print()
print('ship_type_agrupado (categorías finales):')
df['ship_type_agrupado'].value_counts()


Filas marcadas como subcategoría nueva: 754

ship_type_agrupado (categorías finales):


ship_type_agrupado
Bulk carrier                  32171
Container ship                15704
Oil tanker                    15652
General cargo ship            11607
Chemical tanker               11332
Vehicle carrier                3731
Ro-pax ship                    3226
Gas carrier                    2859
LNG carrier                    2567
Other ship types               2539
Ro-ro ship                     1931
Passenger ship                 1436
Refrigerated cargo carrier     1149
Container/ro-ro cargo ship      545
Combination carrier              72
Name: count, dtype: int64

## 3. Atípicos en `te_valor`

`te_valor` es el valor numérico del índice de eficiencia técnica del buque (EIV, EEDI o EEXI,
según `te_metodo`), en gCO₂/tonelada·milla náutica. Antes de usarlo hay que verificar dos cosas:
que la unidad sea comparable entre los tres métodos (si no lo fuera, mezclar sus valores en una
sola columna numérica no tendría sentido), y que no tenga atípicos sin tratar.


In [7]:
# Verificación de unidad: las cadenas originales de technical_efficiency siempre declaran
# la misma unidad (gCO2/t·nm) independientemente del método
for metodo in df['te_metodo'].unique():
    ejemplos = df.loc[df['te_metodo'] == metodo, 'technical_efficiency'].dropna().unique()[:2]
    print(f'{metodo:>16}: {list(ejemplos)}')


  Not Applicable: ['Not Applicable', 'Not Applicable (0 gCO₂/t·nm)']
             EIV: ['EIV (45.57 gCO₂/t·nm)', 'EIV (31.73 gCO₂/t·nm)']
            EEDI: ['EEDI (60.41 gCO₂/t·nm)', 'EEDI (313.8 gCO₂/t·nm)']
     Desconocido: ['0 gCO₂/t·nm', '3.76 gCO₂/t·nm']
            EEXI: ['EEXI (21 gCO₂/t·nm)', 'EEXI (9.47 gCO₂/t·nm)']


In [8]:
print(df['te_valor'].describe())
print()
print('Percentiles altos:')
print(df['te_valor'].quantile([0.9, 0.99, 0.999, 1.0]))


count    106521.000000
mean         16.076964
std         929.449643
min           0.000000
25%           4.190000
50%           6.630000
75%          13.300000
max      208390.000000
Name: te_valor, dtype: float64

Percentiles altos:
0.900        20.050
0.990        39.336
0.999       127.220
1.000    208390.000
Name: te_valor, dtype: float64


La unidad es la misma en los cuatro grupos (incluido "Not Applicable", que también reporta un
valor numérico pese a la etiqueta) — sí es comparable como una sola columna numérica. Pero el
percentil 99,9 es ~261 y el máximo es 208.390: casi 800 veces mayor. Es la misma señal de atípico
estadístico no tratado que ya se vio con horas en mar, CO2 y combustible en el notebook 02 — solo
que en el EDA, `te_valor` se dejó fuera de ese tratamiento (solo se imputaron sus missings
directos, no sus atípicos).

**A diferencia de horas en mar, aquí no hay una cota física evidente** (no hay un límite
regulatorio conocido de antemano para un índice de eficiencia), así que se aplica solo el criterio
estadístico ya validado en el EDA (`atipicosAmissing`, el mismo usado para CO2, combustible y
horas), seguido de imputación por mediana — el mismo criterio, aplicado de forma consistente a la
única variable numérica núcleo que se había quedado sin este paso.


In [9]:
df, n_atipicos_te = tratar_atipicos_te_valor(df)
print(f'Atípicos detectados y tratados en te_valor: {n_atipicos_te}'
      f' ({n_atipicos_te / len(df):.2%} de las filas)')
print()
print(df['te_valor'].describe())


Atípicos detectados y tratados en te_valor: 846 (0.79% de las filas)

count    106521.000000
mean          9.439911
std           7.170033
min           0.000000
25%           4.190000
50%           6.580000
75%          13.040000
max          42.500000
Name: te_valor, dtype: float64


Rango final mucho más razonable (máximo 42,5 gCO₂/t·nm, frente a los 208.390 de antes). Este
tratamiento no cambia ninguna conclusión del EDA (`te_valor` ya se había identificado
como el predictor más débil, V de Cramer 0,055 con `te_metodo` y correlación ≈0 con el target), y
su efecto en el modelo es marginal: es una variable secundaria. Pero sí importa
para dos usos concretos donde sí puede distorsionar: cualquier escalado (StandardScaler) que se
aplique en el notebook 04/05, y la propia interpretabilidad (SHAP) del notebook 07 — un valor 800
veces fuera de rango puede dominar visualmente un gráfico de dependencia aunque no domine el
modelo.


## 4. Variables operativas reconstruidas: distancia navegada y velocidad media

El modelo principal tiene un problema de fondo que no es estadístico sino de contenido: de sus
siete predictores, seis describen **cómo es el buque** (tipo, clase de hielo, índice de eficiencia)
y solo uno describe **qué hizo durante el año** (horas en mar). Para responder a la tercera pregunta
del TFM —simular escenarios operativos de reducción— hace falta algo más que se pueda mover.

El MRV no publica la distancia navegada, pero sí dos intensidades que la contienen:

```
distancia = CO₂ total × 1000 / (kg CO₂ por milla)          ← usa el target
distancia = combustible total × 1000 / (kg combustible por milla)   ← no lo usa
```

**Se usa la segunda**, que no toca la variable objetivo en ningún momento, y la primera sirve solo
para validarla. Que dos caminos aritméticos independientes den el mismo número es la comprobación
de que no se está estimando nada: se está recuperando una magnitud que el armador sí reportó, solo
que de forma implícita. De ahí sale también la velocidad media anual, dividiendo por las horas en
mar.

In [10]:
df, resumen_dist = reconstruir_distancia_velocidad(df, '../data/processed/mrv_consolidado.parquet')

print('Validacion cruzada de las dos vias de reconstruccion:')
print(f"  filas con ambas vias disponibles: {resumen_dist['n_con_ambas_vias']:,}".replace(',', '.'))
print(f"  discrepancia relativa mediana:    {resumen_dist['discrepancia_mediana_pct']:.5f}%")
print(f"  discrepancia relativa p99:        {resumen_dist['discrepancia_p99_pct']:.4f}%")
print(f"  filas con discrepancia < 1%:      {resumen_dist['pct_discrepancia_menor_1pct']:.2f}%")

print(f"\nCobertura: distancia {resumen_dist['cobertura_distancia_pct']:.1f}% | "
      f"velocidad {resumen_dist['cobertura_velocidad_pct']:.1f}%")
print(f"Velocidad media mediana de la flota: {resumen_dist['velocidad_mediana_nudos']:.2f} nudos")

print('\nVelocidad mediana por tipo de buque (prueba de plausibilidad):')
print(df.groupby('ship_type_agrupado')['velocidad_nudos'].median().sort_values(ascending=False).round(2).to_string())

print('\nCobertura de la reconstruccion por tipo de buque:')
cob = df.groupby('ship_type_agrupado')['distancia_nm'].apply(lambda s: 100 * s.notna().mean())
print(cob.sort_values().round(1).to_string())

Validacion cruzada de las dos vias de reconstruccion:
  filas con ambas vias disponibles: 101.028
  discrepancia relativa mediana:    0.00226%
  discrepancia relativa p99:        0.0128%
  filas con discrepancia < 1%:      100.00%

Cobertura: distancia 94.7% | velocidad 94.0%
Velocidad media mediana de la flota: 11.31 nudos

Velocidad mediana por tipo de buque (prueba de plausibilidad):
ship_type_agrupado
Ro-pax ship                   15.83
Refrigerated cargo carrier    15.04
Container/ro-ro cargo ship    14.60
Ro-ro ship                    14.52
Vehicle carrier               14.38
LNG carrier                   14.23
Container ship                13.59
Passenger ship                12.69
Gas carrier                   12.47
Chemical tanker               11.02
Bulk carrier                  10.86
Other ship types              10.77
Oil tanker                    10.68
General cargo ship            10.26
Combination carrier           10.23

Cobertura de la reconstruccion por tipo de buque:


> **Cambio de esquema en la fuente.** **EMSA renombró la columna del ratio en el informe de 2024**,
> quitándole el prefijo `annual_average_`
> (`annual_average_fuel_consumption_per_distance_kg_n_mile` → `fuel_consumption_per_distance_kg_n_mile`).
> Buscando solo el nombre antiguo, la distancia saldría `NaN` entera para **2024 y 2025** —31.072
> filas, el 29% del dataset y 297 Mt de CO₂— **sin lanzar ninguna excepción**, y la cobertura
> caería del 94% al 68%.
>
> Es el mismo tipo de cambio de esquema entre ediciones que obliga a construir `ship_type_agrupado`
> en la sección 2: la fuente cambia de nombres de un año a otro y no avisa. `_combinar_convenciones`
> acepta las dos convenciones y **falla en voz alta** si alguna fila trajera valor en las dos a la
> vez, en vez de dar por supuesto que son excluyentes.
>
> Así el nivel operacional incluye los dos años más recientes, y el simulador (notebook 08) puede
> trabajar sobre la flota de 2025.


La reconstrucción es exacta, no aproximada: las dos vías coinciden con una discrepancia relativa
**mediana del 0,002%**, y el **100% de las filas** queda por debajo del 1%.

La prueba de plausibilidad más convincente es el orden de velocidades por tipo de buque, que no se
ha impuesto en ningún sitio y sale solo: ferris ro-pax y frigoríficos arriba (15-16 nudos),
portacontenedores en torno a 13,7, y graneleros, petroleros y carga general abajo (10,5-11). Es
exactamente el perfil operativo de la flota mercante real, y la mediana global de 11,4 nudos también
lo es.

**Dos limitaciones:**

1. **La cobertura es del ~94% y no es uniforme.** Las dos subcategorías nuevas de EMSA (`Offshore` y
   `Cruise Passenger ship`) son las peor cubiertas, y los cruceros son además de los tipos peor
   predichos por el modelo. El valor ausente se deja como `NaN` a propósito: XGBoost lo trata como
   una rama propia, que es más informativo que imputarlo con una velocidad inventada.
2. **La velocidad media anual no es la velocidad de navegación.** Es distancia entre *horas en mar*,
   y las horas en mar incluyen maniobra, espera y navegación a carga muy baja. Sirve como predictor,
   pero **no** como palanca para simular *slow steaming* — la elasticidad medida en el notebook 04
   lo demuestra, y la conclusión condiciona el diseño del simulador (notebook 08).

Con estas dos variables, los datasets de modelización pasan a tener **tres niveles** en vez de dos,
que es una forma mucho más informativa de presentar el trabajo:

| Nivel | Predictores | Qué pregunta responde |
|---|---|---|
| **principal** | 7 variables de buque y tiempo | Qué factores explican las emisiones con lo que se sabe del buque |
| **operacional** | + distancia y velocidad | Cuánto mejora si además se sabe qué hizo el buque — es el que necesita el simulador |
| **control** | + combustible | Cota superior tautológica, no es un resultado |

## 5. El tamaño del buque: la variable que el MRV no publica

El registro público no publica **ninguna** medida de tamaño del buque: ni arqueo bruto, ni porte,
ni plazas de pasaje. No es una limitación de esta extracción, es el alcance del artículo 21 del
Reglamento (UE) 2015/757. El informe del Centro Común de Investigación de la Comisión sobre este
mismo portal (JRC128870) señala esa ausencia como el límite de lo que se puede analizar con estos
datos, y la literatura que trabaja con el dataset la resuelve **comprando** una base comercial
(Clarksons Shipping Intelligence Network) y cruzando por número IMO.

No hace falta comprar nada. El MRV publica, para cada buque-año, la intensidad **por milla
navegada** y la intensidad **por trabajo de transporte** — y el denominador de la segunda *es* la
magnitud de capacidad que se busca. Dividiendo una entre otra, la capacidad se despeja:

$$\text{capacidad} = \frac{\text{combustible por distancia}\ [\mathrm{kg/milla}] \times 1000}
{\text{combustible por trabajo de transporte}\ [\mathrm{g/(u \cdot milla)}]}$$

Comprobación de unidades: $(\mathrm{g/milla}) / (\mathrm{g/(u \cdot milla)}) = u$.

Es el mismo razonamiento de la sección 4, y con **una ventaja añadida**: la distancia se despeja del
combustible *total*, mientras que la capacidad sale del cociente de dos intensidades, donde el
consumo total se cancela. La capacidad reconstruida no arrastra información sobre el nivel de
consumo del buque, y por eso puede entrar en el nivel `principal` sin acercarlo a la tautología del
nivel `control`.

El Reglamento de Ejecución (UE) 2016/1928 asigna a cada tipo de buque un parámetro de carga
distinto, así que hay **cinco denominadores** posibles y no son intercambiables entre sí: toneladas
de carga, metros cúbicos, porte transportado, pasajeros y toneladas de carga rodada. Se reconstruyen
los cinco por separado.

Como en la sección 4, se usa la vía del **combustible** y la del CO₂ se calcula solo para validar.

In [11]:
df, resumen_cap = reconstruir_capacidad(df, '../data/processed/mrv_consolidado.parquet')

print('Validacion cruzada de las dos vias (combustible vs CO2):')
print(f"  filas con ambas vias disponibles: {resumen_cap['n_con_ambas_vias']:,}".replace(',', '.'))
print(f"  discrepancia relativa mediana:    {resumen_cap['discrepancia_mediana_pct']:.5f}%")
print(f"  discrepancia relativa p99:        {resumen_cap['discrepancia_p99_pct']:.4f}%")
print(f"  filas con discrepancia < 1%:      {resumen_cap['pct_discrepancia_menor_1pct']:.2f}%")

print(f"\nCobertura: capacidad estimada (por buque) {resumen_cap['cobertura_capacidad_estimada_pct']:.1f}% | "
      f"capacidad utilizada (por fila) {resumen_cap['cobertura_capacidad_utilizada_pct']:.1f}%")
print(f"Buques con capacidad reconstruida: {resumen_cap['buques_con_capacidad']:,}".replace(',', '.'))
print('\nCobertura por unidad del Reglamento 2016/1928 (% de filas):')
for unidad, pct in resumen_cap['cobertura_por_unidad_pct'].items():
    print(f'  {unidad:>8}: {pct:5.2f}%')

Validacion cruzada de las dos vias (combustible vs CO2):
  filas con ambas vias disponibles: 104.754
  discrepancia relativa mediana:    0.04124%
  discrepancia relativa p99:        0.4176%
  filas con discrepancia < 1%:      99.63%

Cobertura: capacidad estimada (por buque) 98.2% | capacidad utilizada (por fila) 92.1%
Buques con capacidad reconstruida: 24.415

Cobertura por unidad del Reglamento 2016/1928 (% de filas):
      mass: 86.90%
    volume:  3.19%
       dwt: 13.73%
       pax:  4.36%
   freight:  3.19%


### 5.1 Qué se recupera exactamente (y qué no)

Conviene ser preciso, porque aquí hay un matiz que invalida la lectura ingenua. El documento de
trabajo de la Comisión sobre parámetros de carga define *deadweight carried* como **el
desplazamiento en el puerto de salida menos el peso en rosca**: es la carga realmente embarcada, no
el porte de proyecto del buque. Lo mismo vale para las toneladas y los pasajeros, que son promedios
anuales de lo efectivamente transportado.

De ahí que se construyan **dos variables distintas**, y que no sean intercambiables:

| Variable | Qué es | Nivel al que pertenece |
|---|---|---|
| `capacidad_utilizada` | lo que el buque movió **ese año** | operativa (lo que el buque *hizo*) |
| `capacidad_estimada` | percentil 95 de la serie histórica del buque | atributo del buque (lo que el buque *es*) |

La capacidad de un buque no cambia de un año para otro, así que la dispersión intra-buque es
variación de carga, no de tamaño: la mediana mide la carga típica y el percentil 95 se acerca al
tamaño de diseño. Antes de agregar se descartan los años cuyo valor supera 2,5 veces la mediana del
propio buque — el percentil 99 de ese cociente es 2,0 y el 99,9 es 12,1, así que el corte separa dos
poblaciones bien distintas y afecta al 0,6% de las filas.

**El agregado por buque no cruza la partición**: se calcula solo con filas del mismo `ship_imo_number`,
y la partición de la sección 7 agrupa por buque, así que todos los años de un buque caen del mismo lado.

In [12]:
# Contraste de plausibilidad: capacidad reconstruida por tipo de buque, en su unidad dominante
resumen_tipos = (df.dropna(subset=['capacidad_estimada'])
                   .groupby(['ship_type_agrupado', 'capacidad_unidad'])['capacidad_estimada']
                   .agg(buques='size', mediana='median', p95=lambda s: s.quantile(0.95))
                   .sort_values('buques', ascending=False)
                   .head(12)
                   .round(0))
resumen_tipos

,,buques,mediana,p95
ship_type_agrupado,capacidad_unidad,,,
Bulk carrier,mass,27795,42216.0,126527.0
Container ship,mass,15093,33688.0,162919.0
Oil tanker,mass,14634,65435.0,174993.0
Chemical tanker,mass,10805,20503.0,38633.0
General cargo ship,dwt,10356,7964.0,37700.0
Vehicle carrier,mass,3482,7365.0,10864.0
Ro-pax ship,pax,2853,989.0,7147.0
Gas carrier,mass,2659,8625.0,43516.0
LNG carrier,volume,2446,91507.0,143599.0


### 5.2 Contraste con buques identificables

La prueba de plausibilidad de la sección 4 (el orden de velocidades por tipo de buque sale solo)
tiene aquí una versión mucho más exigente, porque el dataset trae el **nombre** del buque: se puede
comparar la capacidad reconstruida contra la capacidad publicada de buques concretos y conocidos.

In [13]:
identificables = [
    'SHAGRA',                  # metanero clase Q-Max, 266.000 m3 de capacidad publicada
    'UMM SLAL',                # metanero clase Q-Max, 266.000 m3
    'CMA CGM JACQUES SAADE',   # portacontenedores, 220.766 t de porte
    'MSC WORLD EUROPA',        # crucero, 6.762 pasajeros maximos
    'SYMPHONY OF THE SEAS',    # crucero, 6.680 pasajeros maximos
    'OASIS OF THE SEAS',       # crucero, 6.780 pasajeros maximos
    'AIDANOVA',                # crucero, ~6.600 pasajeros maximos
    'COSTA TOSCANA',           # crucero, 6.730 pasajeros maximos
    'AIDABELLA',               # crucero mediano, ~2.500 pasajeros maximos
]
contraste = (df[df['ship_name'].isin(identificables)]
             .dropna(subset=['capacidad_estimada'])
             .groupby(['ship_name', 'capacidad_unidad'])['capacidad_estimada']
             .first()
             .round(0)
             .reset_index()
             .rename(columns={'capacidad_estimada': 'capacidad_reconstruida'}))
contraste

,ship_name,capacidad_unidad,capacidad_reconstruida
0,AIDABELLA,pax,2195.0
1,AIDANOVA,pax,6256.0
2,CMA CGM JACQUES SAADE,mass,209777.0
3,COSTA TOSCANA,pax,5945.0
4,MSC WORLD EUROPA,pax,6027.0
5,OASIS OF THE SEAS,pax,5924.0
6,SHAGRA,volume,242267.0
7,SYMPHONY OF THE SEAS,pax,5885.0
8,UMM SLAL,volume,230746.0


Los órdenes de magnitud y las unidades salen solos, sin haberlos impuesto en ningún sitio: los dos
metaneros de la clase Q-Max se reconstruyen en metros cúbicos y cerca de su capacidad publicada, el
portacontenedores en toneladas cerca de su porte, y los cruceros en pasajeros por debajo pero cerca
de su pasaje máximo — que es lo esperable, porque ningún crucero navega el año entero al 100% de
ocupación. Y el crucero mediano de la lista se separa del resto en el mismo factor en que se separa
en la realidad.

**Esto no sustituye a una validación externa formal.** Es un contraste sobre una muestra elegida,
no una medición de error. La validación contra una fuente independiente de particulares de buque
(IMO GISIS o Equasis, consulta manual sobre una muestra aleatoria) queda como línea abierta y se
declara así en la memoria.

## 6. `total_fuel_consumption_m_tonnes`: casi tautológico con el target

`total_co2_emissions_m_tonnes` y `total_fuel_consumption_m_tonnes` están relacionados por el
factor de emisión del fuel marino, prácticamente constante (verificado en la ingesta, 3,11-3,21
t/t en los 8 años). Se recalcula aquí sobre los datos ya depurados para dejarlo explícito en este
notebook.


In [14]:
ratio = df['total_co2_emissions_m_tonnes'] / df['total_fuel_consumption_m_tonnes']
print(ratio.describe())


count    100883.000000
mean          3.132676
std           0.066916
min           2.508379
25%           3.124569
50%           3.136619
75%           3.157126
max           3.288215
dtype: float64


El ratio apenas se mueve: percentil 25 y 75 casi calcados a la media. **Predecir CO2 a partir del
combustible no es modelización real, es casi una multiplicación por una constante.** Si se incluye
como predictor principal, el modelo "hace trampa": la pregunta de negocio del TFM ("qué factores
operativos influyen en las emisiones") deja de tener respuesta real, porque el modelo se apoyaría
en una relación que no aporta explicatividad.

**Decisión:** se generan dos versiones del dataset de modelización:
- **Principal** (sin `total_fuel_consumption_m_tonnes`): responde a la pregunta real del TFM — qué
  combinación de variables operativas y estructurales predice las emisiones. Es el modelo de
  cabecera, el que se usa en interpretabilidad y en el simulador de escenarios.
- **Control** (con `total_fuel_consumption_m_tonnes`): se entrena aparte y se presenta en la
  memoria como cota superior de referencia ("con el combustible, así de bien se podría
  predecir"), etiquetado explícitamente como modelo de control, no como resultado principal.

Los predictores del modelo principal (variables operativas y estructurales, ya con las decisiones
de las secciones 1 y 2 aplicadas):


In [15]:
print('Predictores del modelo principal:')
for p in PREDICTORES_PRINCIPAL:
    print(' -', p)
print()
print('Predictores del modelo de control (añade):', set(PREDICTORES_CONTROL) - set(PREDICTORES_PRINCIPAL))


Predictores del modelo principal:
 - time_spent_at_sea_hours
 - ship_type_agrupado
 - es_subcategoria_nueva
 - ice_class
 - te_metodo
 - te_valor
 - prop_missings

Predictores del modelo de control (añade): {'total_fuel_consumption_m_tonnes'}


## 7. Partición train/test agrupada por buque

El dataset es un panel: el mismo buque (`ship_imo_number`) aparece en varias filas, una por año.
Un split aleatorio por fila dejaría el mismo buque repartido entre train y test — el modelo podría
aprender a "reconocer" el nivel base de emisiones de ese buque concreto en vez de generalizar el
patrón a partir de sus variables operativas. Es una fuga de información a nivel de entidad, no de
tiempo, pero igual de real.

**Decisión:** partición 80/20 agrupada por `ship_imo_number` (`GroupShuffleSplit`), de forma que
todas las filas de un mismo buque caen enteras en train o en test, y la serie histórica completa
de cada buque queda en un único lado del split.


In [16]:
df = dividir_train_test(df)

n_train, n_test = df['es_train'].sum(), (~df['es_train']).sum()
print(f'Train: {n_train:,} filas ({n_train / len(df):.1%})'.replace(',', '.'))
print(f'Test:  {n_test:,} filas ({n_test / len(df):.1%})'.replace(',', '.'))

buques_train = set(df.loc[df['es_train'], 'ship_imo_number'])
buques_test = set(df.loc[~df['es_train'], 'ship_imo_number'])
print(f"Buques en ambos lados del split (debe ser 0): {len(buques_train & buques_test)}")


Train: 85.416 filas (80.2%)
Test:  21.105 filas (19.8%)
Buques en ambos lados del split (debe ser 0): 0


## 8. Construcción de los datasets de modelización

Se generan cuatro tablas de salida:

- `principal_raw` / `control_raw`: categóricas sin codificar (`ship_type_agrupado`, `ice_class`,
  `te_metodo`, `prop_missings`), pensadas para el notebook 05 (deep learning), donde puede
  interesar usar capas de *embeddings* en vez de one-hot.
- `principal_ml` / `control_ml`: mismas variables con codificación one-hot (`drop_first=True`,
  mismo criterio que `crear_data_modelo` de `src/FuncionesMineria.py`), listas para regresión
  regularizada, GLM, Random Forest o Gradient Boosting en el notebook 04.

Todas incluyen las columnas identificadoras, el target y la columna `es_train` de la sección
anterior, para que cualquier notebook posterior reconstruya la misma partición sin recalcularla.
No se aplica ningún escalado aquí: el escalado se ajusta solo sobre train en el notebook que
corresponda, para no filtrar estadísticos del test en el propio preprocesado.


In [17]:
datasets = construir_datasets_modelizacion(df)

# company_name tiene NaN legítimos (69% missing, ver EDA). En los datasets 'operacional' también son
# legítimos los NaN de distancia_nm y velocidad_nudos: son las filas donde el MRV no publica las
# intensidades por milla (~31%). Se dejan sin imputar a propósito (ver sección 4).
opcionales = (['company_name', 'distancia_nm', 'velocidad_nudos',
               'capacidad_utilizada', 'capacidad_unidad'] + CAPACIDAD_COLS)
for nombre, tabla in datasets.items():
    n_nulos = tabla.drop(columns=[c for c in opcionales if c in tabla.columns]).isna().sum().sum()
    n_sin_vel = int(tabla['velocidad_nudos'].isna().sum()) if 'velocidad_nudos' in tabla.columns else 0
    extra = f' | sin velocidad: {n_sin_vel:,}'.replace(',', '.') if n_sin_vel else ''
    print(f'{nombre:16s} {tabla.shape[0]:>7,} filas x {tabla.shape[1]:>3} columnas'
          f' | nulos inesperados: {n_nulos}{extra}'.replace(',', '.'))

principal_raw    106.521 filas x  14 columnas | nulos inesperados: 0
principal_ml     106.521 filas x  41 columnas | nulos inesperados: 0
tamano_raw       106.521 filas x  20 columnas | nulos inesperados: 0
tamano_ml        106.521 filas x  47 columnas | nulos inesperados: 0
operacional_raw  106.521 filas x  16 columnas | nulos inesperados: 0 | sin velocidad: 6.434
operacional_ml   106.521 filas x  43 columnas | nulos inesperados: 0 | sin velocidad: 6.434
operacional_tamano_raw 106.521 filas x  23 columnas | nulos inesperados: 0 | sin velocidad: 6.434
operacional_tamano_ml 106.521 filas x  50 columnas | nulos inesperados: 0 | sin velocidad: 6.434
control_raw      106.521 filas x  15 columnas | nulos inesperados: 0
control_ml       106.521 filas x  42 columnas | nulos inesperados: 0


In [18]:
datasets['principal_ml'].head(3)


,reporting_year,ship_imo_number,ship_name,company_name,total_co2_emissions_m_tonnes,es_train,target_reconstruido,time_spent_at_sea_hours,es_subcategoria_nueva,te_valor,ship_type_agrupado_Chemical tanker,ship_type_agrupado_Combination carrier,ship_type_agrupado_Container ship,ship_type_agrupado_Container/ro-ro cargo ship,ship_type_agrupado_Gas carrier,ship_type_agrupado_General cargo ship,ship_type_agrupado_LNG carrier,ship_type_agrupado_Oil tanker,ship_type_agrupado_Other ship types,ship_type_agrupado_Passenger ship,ship_type_agrupado_Refrigerated cargo carrier,ship_type_agrupado_Ro-pax ship,ship_type_agrupado_Ro-ro ship,ship_type_agrupado_Vehicle carrier,ice_class_IA Super,ice_class_IB,ice_class_IC,ice_class_PC1,ice_class_PC2,ice_class_PC3,ice_class_PC4,ice_class_PC5,ice_class_PC6,ice_class_PC7,ice_class_Sin clase de hielo,te_metodo_EEDI,te_metodo_EEXI,te_metodo_EIV,te_metodo_Not Applicable,prop_missings_0.25,prop_missings_0.5
0,2018,5383304,ASTORIA,NaN,20080.2509,False,False,4170.20,0,4.52,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,True,False
1,2018,6417097,MARCO POLO,NaN,25689.0258,True,False,4360.57,0,8.65,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,True,False
2,2018,6511128,RED STAR 1,NaN,6941.3412,True,False,2712.00,0,6.58,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False


Verificación rápida de sanidad: ninguna columna de predictores tiene nulos (`company_name` sí,
pero es un identificador que no entra como predictor, no una variable del modelo — su 69% de
missing ya está documentado y es de tipo no aleatorio: no todos los informes declaran la compañía
gestora).


## 9. Guardado de los datasets de salida


In [19]:
import os
os.makedirs('../data/processed', exist_ok=True)

rutas = {}
for nombre, tabla in datasets.items():
    ruta = f'../data/processed/mrv_features_{nombre}.parquet'
    tabla.to_parquet(ruta, index=False)
    rutas[nombre] = ruta
    print(f'Guardado: {ruta}  ({tabla.shape[0]:,} x {tabla.shape[1]})'.replace(',', '.'))


Guardado: ../data/processed/mrv_features_principal_raw.parquet  (106.521 x 14)


Guardado: ../data/processed/mrv_features_principal_ml.parquet  (106.521 x 41)
Guardado: ../data/processed/mrv_features_tamano_raw.parquet  (106.521 x 20)


Guardado: ../data/processed/mrv_features_tamano_ml.parquet  (106.521 x 47)


Guardado: ../data/processed/mrv_features_operacional_raw.parquet  (106.521 x 16)


Guardado: ../data/processed/mrv_features_operacional_ml.parquet  (106.521 x 43)


Guardado: ../data/processed/mrv_features_operacional_tamano_raw.parquet  (106.521 x 23)


Guardado: ../data/processed/mrv_features_operacional_tamano_ml.parquet  (106.521 x 50)


Guardado: ../data/processed/mrv_features_control_raw.parquet  (106.521 x 15)
Guardado: ../data/processed/mrv_features_control_ml.parquet  (106.521 x 42)


## 10. Resumen y siguiente paso

**Decisiones tomadas en este notebook:**

1. Verificación de la fiabilidad del target (relación física CO₂/combustible): 0 filas con el ratio
   fuera de rango con el tratamiento de atípicos del notebook 02. Se excluyen 180 filas (0,17%)
   cuya pareja CO₂/combustible es incompatible ya en el dato crudo.
2. `ship_type_agrupado` (subcategorías 2023-2025 fusionadas con su padre) + `es_subcategoria_nueva`
   (flag para analizar su efecto propio en interpretabilidad).
3. `te_valor` tratado de atípicos con el mismo criterio estadístico del EDA. Aquí sí procede el criterio estadístico: `te_valor`
   es un índice de eficiencia, no una magnitud de tamaño, y un valor de 208.390 gCO₂/t·nm frente a
   un percentil 99,9 de 127 no es un buque grande, es un error de codificación.
4. **Distancia navegada y velocidad media reconstruidas** a partir de las intensidades por milla
   que sí publica el MRV, por la vía del combustible (que no usa el target) y validadas contra la
   vía del CO₂: coinciden con una discrepancia mediana del 0,002%. Cobertura del 94%, sin imputar.
5. **Tres** niveles de dataset de modelización en vez de dos — principal (variables de buque y
   tiempo), operacional (+ distancia y velocidad, el que necesita el simulador) y control
   (+ combustible, cota superior tautológica) — cada uno en versión `raw` (para deep
   learning/embeddings) y `ml` (one-hot, para ML clásico), con 106.521 filas.
6. Partición train/test 80/20 agrupada por buque.

**Siguiente paso:** notebook 04 — modelización con ML clásico (regresión regularizada, Random
Forest, Gradient Boosting, XGBoost), comparando el dataset principal frente al de control, con
`lm`, `lm_forward/backward/stepwise`, `Rsq` y `validacion_cruzada_lm` de `src/FuncionesMineria.py`
como modelo lineal de referencia con selección de variables.